# Trening YOLOv8x-pose — `football-field-detection-16_extended`

Notatnik przeznaczony do **Google Colab**.

Cel eksperymentu:

- rozpocząć uczenie ponownie od bazowych wag `yolov8x-pose.pt`,
- wykorzystać rozszerzony zbiór `football-field-detection-16_extended`,
- zachować najważniejsze ustawienia wcześniejszego treningu:
  - `imgsz=640`,
  - `batch=48`,
  - `epochs=500`,
- trenować w Google Colab na GPU (docelowo NVIDIA A100),
- zapisać cały run oraz `best.pt` na Google Drive.

### Ważne metodologicznie

To **nie jest kontynuacja treningu** ze starego `trained_keypoints.pt`.
Model startuje ponownie od `yolov8x-pose.pt`, dzięki czemu porównanie starego i nowego modelu dotyczy przede wszystkim zmiany zbioru treningowego.

Rozszerzony dataset zawiera obrazy zapisane fizycznie jako `640×640 Stretch`, zgodnie z preprocessingiem zastosowanym w bazowym `football-field-detection-16`.


## 0. Przed uruchomieniem

W Colab ustaw:

**Runtime → Change runtime type → GPU**

Jeżeli masz możliwość wyboru akceleratora, wybierz **A100**.

Na Google Drive powinien znajdować się plik:

```text
football-field-detection-16_extended.zip
```

Może być w dowolnym podfolderze `MyDrive` — notatnik spróbuje znaleźć go automatycznie.


In [1]:
# Montowanie Google Drive
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# Instalacja bibliotek
!pip install -q ultralytics pyyaml

import os
import json
import shutil
import platform
from pathlib import Path

import cv2
import yaml
import numpy as np
import matplotlib.pyplot as plt
import torch
import ultralytics

from ultralytics import YOLO

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")
else:
    raise RuntimeError("Brak GPU CUDA. Włącz GPU w ustawieniach Colaba.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 kB 8.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Python: 3.13.15
PyTorch: 2.11.0+cu128
Ultralytics: 8.4.143
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
VRAM: 39.5 GB


## 1. Konfiguracja eksperymentu

Jeżeli ZIP na Drive ma inną nazwę, zmień tylko `ZIP_FILENAME`.

Parametry treningowe odpowiadają wcześniejszemu treningowi modelu punktów charakterystycznych.
Pozostałych hiperparametrów nie nadpisujemy ręcznie — pozostają zgodne z domyślną konfiguracją używanej wersji Ultralytics.


In [3]:
# =============================================================================
# KONFIGURACJA
# =============================================================================

ZIP_FILENAME = "football-field-detection-16_extended.zip"

DRIVE_ROOT = Path("/content/drive/MyDrive")
WORK_ROOT = Path("/content/keypoints_training_extended")
EXTRACT_ROOT = WORK_ROOT / "dataset"

# Wyniki będą zapisywane bezpośrednio na Google Drive.
DRIVE_RESULTS_ROOT = DRIVE_ROOT / "CvFootballTracker_training_results"
RUN_NAME = "keypoints_extended_yolov8x_pose"

# Bazowy model — NIE stare trained_keypoints.pt.
BASE_MODEL = "yolov8x-pose.pt"

# Parametry zgodne z poprzednim treningiem.
IMGSZ = 640
BATCH = 48
EPOCHS = 500

WORK_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print("BASE_MODEL:", BASE_MODEL)
print("IMGSZ:", IMGSZ)
print("BATCH:", BATCH)
print("EPOCHS:", EPOCHS)
print("RUN_NAME:", RUN_NAME)
print("Wyniki Drive:", DRIVE_RESULTS_ROOT)

BASE_MODEL: yolov8x-pose.pt
IMGSZ: 640
BATCH: 48
EPOCHS: 500
RUN_NAME: keypoints_extended_yolov8x_pose
Wyniki Drive: /content/drive/MyDrive/CvFootballTracker_training_results


In [4]:
# =============================================================================
# AUTOMATYCZNE ODNALEZIENIE ZIP-A NA GOOGLE DRIVE
# =============================================================================

matches = list(DRIVE_ROOT.rglob(ZIP_FILENAME))

if not matches:
    raise FileNotFoundError(
        f"Nie znaleziono {ZIP_FILENAME} w {DRIVE_ROOT}. "
        "Zmień ZIP_FILENAME albo sprawdź, czy plik został wrzucony na Drive."
    )

if len(matches) > 1:
    print("Znaleziono kilka plików o tej nazwie:")
    for i, p in enumerate(matches):
        print(i, p)
    print("\nUżywam pierwszego wyniku.")

ZIP_PATH = matches[0]

print("ZIP:", ZIP_PATH)

ZIP: /content/drive/MyDrive/CvFootballTracker_Data/football-field-detection-16_extended.zip


In [5]:
# =============================================================================
# ROZPAKOWANIE DATASETU NA LOKALNY DYSK COLABA
# =============================================================================

if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)

EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

print("Rozpakowywanie...")
shutil.unpack_archive(str(ZIP_PATH), str(EXTRACT_ROOT))
print("Gotowe:", EXTRACT_ROOT)

# Szukamy data.yaml niezależnie od tego, czy ZIP zawiera nadrzędny folder.
yaml_candidates = list(EXTRACT_ROOT.rglob("data.yaml"))

if not yaml_candidates:
    raise FileNotFoundError("Po rozpakowaniu nie znaleziono data.yaml.")

print("\nZnalezione data.yaml:")
for p in yaml_candidates:
    print(" -", p)

# Preferujemy data.yaml należący do extended datasetu.
preferred = [
    p for p in yaml_candidates
    if "football-field-detection-16_extended" in str(p.parent)
]

SOURCE_DATA_YAML = preferred[0] if preferred else yaml_candidates[0]
DATASET_ROOT = SOURCE_DATA_YAML.parent

print("\nDataset root:", DATASET_ROOT)
print("Source YAML:", SOURCE_DATA_YAML)

Rozpakowywanie...
Gotowe: /content/keypoints_training_extended/dataset

Znalezione data.yaml:
 - /content/keypoints_training_extended/dataset/football-field-detection-16_extended/data.yaml

Dataset root: /content/keypoints_training_extended/dataset/football-field-detection-16_extended
Source YAML: /content/keypoints_training_extended/dataset/football-field-detection-16_extended/data.yaml


## 2. Poprawienie `data.yaml` pod środowisko Colab

Builder datasetu był uruchamiany na Windowsie, więc pole `path:` w ZIP-ie może wskazywać lokalną ścieżkę Windows.

Nie zmieniamy zawartości datasetu. Tworzymy jedynie kopię YAML-a z poprawną ścieżką dla bieżącego środowiska Colab.


In [6]:
# =============================================================================
# DATA.YAML DLA COLABA
# =============================================================================

with open(SOURCE_DATA_YAML, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

data_cfg["path"] = str(DATASET_ROOT)
data_cfg["train"] = "train/images"
data_cfg["val"] = "valid/images"

# Test nie jest potrzebny do treningu.
# Jeżeli nie istnieje w extended dataset, usuwamy stare odwołanie.
if "test" in data_cfg:
    test_path = DATASET_ROOT / str(data_cfg["test"])
    if not test_path.exists():
        data_cfg.pop("test", None)

COLAB_DATA_YAML = WORK_ROOT / "data_colab.yaml"

with open(COLAB_DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        data_cfg,
        f,
        sort_keys=False,
        allow_unicode=True
    )

print(COLAB_DATA_YAML.read_text(encoding="utf-8"))

path: /content/keypoints_training_extended/dataset/football-field-detection-16_extended
train: train/images
val: valid/images
kpt_shape:
- 32
- 3
flip_idx:
- 24
- 25
- 26
- 27
- 28
- 29
- 22
- 23
- 21
- 17
- 18
- 19
- 20
- 13
- 14
- 15
- 16
- 9
- 10
- 11
- 12
- 8
- 6
- 7
- 0
- 1
- 2
- 3
- 4
- 5
- 31
- 30
names:
- pitch
nc: 1
roboflow:
  license: CC BY 4.0
  project: football-field-detection-f07vi
  url: https://universe.roboflow.com/roboflow-jvuqo/football-field-detection-f07vi/dataset/16
  version: 16
  workspace: roboflow-jvuqo



## 3. Kontrola zbioru przed treningiem

Ta sekcja sprawdza:

- liczbę obrazów train/valid,
- obecność odpowiadających labeli,
- format YOLO Pose,
- `kpt_shape=[32,3]`,
- czy przykładowe labele mają dokładnie 32 keypointy.

Jeżeli coś jest niepoprawne, trening nie powinien być uruchamiany.


In [7]:
# =============================================================================
# WALIDACJA STRUKTURY DATASETU
# =============================================================================

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def list_images(path: Path):
    return sorted(
        p for p in path.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    )

def check_split(split_name: str):
    images_dir = DATASET_ROOT / split_name / "images"
    labels_dir = DATASET_ROOT / split_name / "labels"

    assert images_dir.exists(), f"Brak {images_dir}"
    assert labels_dir.exists(), f"Brak {labels_dir}"

    images = list_images(images_dir)
    missing_labels = []

    for img in images:
        label = labels_dir / f"{img.stem}.txt"
        if not label.exists():
            missing_labels.append(img.name)

    print(
        f"{split_name}: images={len(images)}, "
        f"missing_labels={len(missing_labels)}"
    )

    if missing_labels:
        print("Pierwsze brakujące:", missing_labels[:10])
        raise RuntimeError(f"Brak labeli w {split_name}.")

    return images, labels_dir

train_images, train_labels_dir = check_split("train")
valid_images, valid_labels_dir = check_split("valid")

assert data_cfg.get("kpt_shape") == [32, 3], (
    f"Nieoczekiwane kpt_shape: {data_cfg.get('kpt_shape')}"
)

print("\nkpt_shape:", data_cfg.get("kpt_shape"))
print("names:", data_cfg.get("names"))

EXPECTED_VALUES = 5 + 32 * 3  # class + bbox4 + 32*(x,y,v)

def validate_labels(images, labels_dir, split):
    bad = []

    for img in images:
        label_path = labels_dir / f"{img.stem}.txt"
        lines = [
            line.strip()
            for line in label_path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]

        if len(lines) != 1:
            bad.append((img.name, f"liczba obiektów={len(lines)}"))
            continue

        n_values = len(lines[0].split())

        if n_values != EXPECTED_VALUES:
            bad.append(
                (img.name, f"wartości={n_values}, oczekiwano={EXPECTED_VALUES}")
            )

    print(f"{split}: błędne labele={len(bad)}")

    if bad:
        print("Pierwsze błędy:", bad[:10])
        raise RuntimeError(f"Wykryto niepoprawne labele w {split}.")

validate_labels(train_images, train_labels_dir, "train")
validate_labels(valid_images, valid_labels_dir, "valid")

print("\nDataset wygląda poprawnie.")

train: images=696, missing_labels=0
valid: images=93, missing_labels=0

kpt_shape: [32, 3]
names: ['pitch']
train: błędne labele=0
valid: błędne labele=0

Dataset wygląda poprawnie.


## 4. Kontrolna wizualizacja GT

Pokazujemy kilka przykładowych obrazów z części treningowej wraz z keypointami posiadającymi `visibility > 0`.

To jest ostatni sanity check przed uruchomieniem kosztownego treningu.


In [8]:
# =============================================================================
# WIZUALIZACJA LOSOWYCH GT
# =============================================================================

rng = np.random.default_rng(42)

sample_count = min(6, len(train_images))
sample_ids = rng.choice(
    len(train_images),
    size=sample_count,
    replace=False
)

for idx in sample_ids:
    image_path = train_images[int(idx)]
    label_path = train_labels_dir / f"{image_path.stem}.txt"

    image_bgr = cv2.imread(str(image_path))
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    h, w = image_bgr.shape[:2]

    values = np.asarray(
        [float(x) for x in label_path.read_text().strip().split()],
        dtype=np.float64
    )

    kp = values[5:].reshape(32, 3)

    xy = kp[:, :2].copy()
    xy[:, 0] *= w
    xy[:, 1] *= h

    visible = kp[:, 2] > 0
    ids = np.flatnonzero(visible)

    plt.figure(figsize=(8, 8))
    plt.imshow(image_rgb)

    if len(ids):
        plt.scatter(xy[ids, 0], xy[ids, 1], s=30)

        for kp_id in ids:
            plt.annotate(
                f"K{kp_id:02d}",
                (xy[kp_id, 0], xy[kp_id, 1]),
                xytext=(3, 3),
                textcoords="offset points",
                fontsize=8
            )

    plt.title(
        f"{image_path.name} | visible GT: {len(ids)}/32"
    )
    plt.axis("off")
    plt.show()

Output hidden; open in https://colab.research.google.com to view.

# 5. Trening

Model startuje od:

```text
yolov8x-pose.pt
```

a nie od starego `trained_keypoints.pt`.

Ustawiamy wyłącznie parametry, które chcemy zachować identyczne względem poprzedniego treningu:

```text
imgsz = 640
batch = 48
epochs = 500
```

Pozostałe parametry pozostają w konfiguracji domyślnej biblioteki Ultralytics, tak aby nie wprowadzać dodatkowych zmian w eksperymencie.


In [9]:
# =============================================================================
# INICJALIZACJA MODELU
# =============================================================================

model = YOLO(BASE_MODEL)

print(model)

YOLO(
  (model): PoseModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 80, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(80, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(80, 160, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(160, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(160, 160, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(160, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(400, 160, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(160, eps=0.001, momentum=0.03, affine=True, track_runnin

In [10]:
# =============================================================================
# START TRENINGU
# =============================================================================
#
# UWAGA:
# To może potrwać długo. Przed uruchomieniem upewnij się, że:
# 1) GT z poprzedniej sekcji wygląda poprawnie,
# 2) Colab używa GPU,
# 3) najlepiej masz przydzielone A100.
#
# Wyniki są zapisywane bezpośrednio na Google Drive.

train_results = model.train(
    data=str(COLAB_DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=str(DRIVE_RESULTS_ROOT),
    name=RUN_NAME,
    exist_ok=False,
)

print("Trening zakończony.")

Ultralytics 8.4.143 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=48, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/keypoints_training_extended/data_colab.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8x-pose.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=keypo

## 6. Lokalizacja najlepszych wag

Ultralytics zapisuje:

```text
weights/best.pt
weights/last.pt
```

Do dalszej ewaluacji wykorzystujemy `best.pt`, analogicznie jak w poprzednim eksperymencie.


In [11]:
# =============================================================================
# ODNALEZIENIE BEST.PT
# =============================================================================

RUN_DIR = DRIVE_RESULTS_ROOT / RUN_NAME
BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"

assert BEST_PT.exists(), f"Brak {BEST_PT}"

print("Run:", RUN_DIR)
print("BEST:", BEST_PT)
print("LAST:", LAST_PT if LAST_PT.exists() else "brak")

# Dodatkowa kopia o jednoznacznej nazwie.
FINAL_WEIGHTS = DRIVE_RESULTS_ROOT / "trained_keypoints_extended.pt"
shutil.copy2(BEST_PT, FINAL_WEIGHTS)

print("\nGotowy model:")
print(FINAL_WEIGHTS)

Run: /content/drive/MyDrive/CvFootballTracker_training_results/keypoints_extended_yolov8x_pose
BEST: /content/drive/MyDrive/CvFootballTracker_training_results/keypoints_extended_yolov8x_pose/weights/best.pt
LAST: /content/drive/MyDrive/CvFootballTracker_training_results/keypoints_extended_yolov8x_pose/weights/last.pt

Gotowy model:
/content/drive/MyDrive/CvFootballTracker_training_results/trained_keypoints_extended.pt


## 7. Końcowa walidacja `best.pt`

Walidacja wykorzystuje ten sam `data.yaml`, `imgsz=640` oraz `batch=48`.


In [12]:
# =============================================================================
# FINAL VALIDATION
# =============================================================================

best_model = YOLO(str(BEST_PT))

val_results = best_model.val(
    data=str(COLAB_DATA_YAML),
    imgsz=IMGSZ,
    batch=BATCH,
)

print(val_results)

Ultralytics 8.4.143 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLOv8x-pose summary (fused): 121 layers, 69,784,275 parameters, 0 gradients, 264.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2099.3±447.8 MB/s, size: 75.5 KB)
val: Scanning /content/keypoints_training_extended/dataset/football-field-detection-16_extended/valid/labels.cache... 93 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 93/93 35.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.1it/s 1.8s
                   all         93         93      0.999          1      0.995      0.967      0.859      0.828      0.781      0.699
Speed: 2.6ms preprocess, 5.9ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to /content/runs/pose/val
ultralytics.utils.metrics.PoseMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.uti

## 8. Podgląd predykcji modelu na valid

Kilka przykładowych obrazów z części walidacyjnej wraz z predykcjami `best.pt`.


In [13]:
# =============================================================================
# PRZYKŁADOWE PREDYKCJE
# =============================================================================

rng = np.random.default_rng(123)

sample_count = min(6, len(valid_images))
sample_ids = rng.choice(
    len(valid_images),
    size=sample_count,
    replace=False
)

for idx in sample_ids:
    image_path = valid_images[int(idx)]

    result = best_model.predict(
        source=str(image_path),
        imgsz=IMGSZ,
        verbose=False,
    )[0]

    plotted_bgr = result.plot()
    plotted_rgb = cv2.cvtColor(plotted_bgr, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(8, 8))
    plt.imshow(plotted_rgb)
    plt.title(image_path.name)
    plt.axis("off")
    plt.show()

Output hidden; open in https://colab.research.google.com to view.

## 9. Zapis konfiguracji eksperymentu

Tworzymy mały plik JSON z najważniejszymi informacjami potrzebnymi później do opisania eksperymentu w pracy.


In [14]:
# =============================================================================
# ZAPIS KONFIGURACJI DO DRIVE
# =============================================================================

experiment_info = {
    "experiment": RUN_NAME,
    "base_model": BASE_MODEL,
    "dataset_zip": str(ZIP_PATH),
    "dataset_root_colab": str(DATASET_ROOT),
    "data_yaml_colab": str(COLAB_DATA_YAML),
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "pytorch_version": torch.__version__,
    "ultralytics_version": ultralytics.__version__,
    "train_images": len(train_images),
    "valid_images": len(valid_images),
    "best_weights": str(BEST_PT),
    "final_weights_copy": str(FINAL_WEIGHTS),
    "training_strategy": (
        "fresh transfer-learning run initialized from yolov8x-pose.pt; "
        "not resumed from the previous trained_keypoints.pt"
    ),
}

CONFIG_PATH = RUN_DIR / "experiment_config.json"

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(
        experiment_info,
        f,
        indent=2,
        ensure_ascii=False
    )

print(json.dumps(experiment_info, indent=2, ensure_ascii=False))
print("\nSaved:", CONFIG_PATH)

{
  "experiment": "keypoints_extended_yolov8x_pose",
  "base_model": "yolov8x-pose.pt",
  "dataset_zip": "/content/drive/MyDrive/CvFootballTracker_Data/football-field-detection-16_extended.zip",
  "dataset_root_colab": "/content/keypoints_training_extended/dataset/football-field-detection-16_extended",
  "data_yaml_colab": "/content/keypoints_training_extended/data_colab.yaml",
  "epochs": 500,
  "imgsz": 640,
  "batch": 48,
  "gpu": "NVIDIA A100-SXM4-40GB",
  "pytorch_version": "2.11.0+cu128",
  "ultralytics_version": "8.4.143",
  "train_images": 696,
  "valid_images": 93,
  "best_weights": "/content/drive/MyDrive/CvFootballTracker_training_results/keypoints_extended_yolov8x_pose/weights/best.pt",
  "final_weights_copy": "/content/drive/MyDrive/CvFootballTracker_training_results/trained_keypoints_extended.pt",
  "training_strategy": "fresh transfer-learning run initialized from yolov8x-pose.pt; not resumed from the previous trained_keypoints.pt"
}

Saved: /content/drive/MyDrive/CvFoot

# Gotowe

Do dalszej ewaluacji użyj:

```text
trained_keypoints_extended.pt
```

z katalogu:

```text
MyDrive/CvFootballTracker_training_results/
```

Następnie należy uruchomić **dokładnie tę samą ewaluację 100 zamrożonych klatek SoccerNet-Calibration**, którą wykonano dla starego modelu. Dzięki temu będzie można bezpośrednio porównać wpływ rozszerzenia zbioru treningowego.
